# Analyzing Flood Impact on Infrastructure

In [7]:
# Import libraries
import ee
import geemap
import pandas as pd
import geopandas as gpd

import os

## . Connect to GEE

In [9]:
# Authenticate GEE
ee.Authenticate()

#Initialize GEE
ee.Initialize()

## . Flood Parameters

In [10]:
# Pre-flood date
before_start = "2022-08-01"
before_end   = "2022-09-10"

# Flood date
after_start = "2022-09-14"
after_end   = "2022-11-18"

# Cloud percentage value
cloud_perc = 58

# Radius for speckle filtering (Sentinel-1)
speckle_radius = 50 # Unit is meter

flood_threshold = -15


## . Visualization Parameters

In [40]:
# Visualization Parameters for Boundary
vis_params_aoi = {'color': 'red', 'fillColor': '00000000'}

# Visualization Parameters for Sentinel-2 (True Colour - RGB)
vis_params_s2_rgb = {"min": 0.1, "max": 0.6, "bands": ["B4", "B3", "B2"], "gamma": 0.9}

# Visualization Parameters for Sentinel-2 (False colour)
vis_params_s2_fcc = {"min": 0.01, "max": 0.6, "bands": ["B8", "B4", "B3"], "gamma": 0.9}

# Visualization Parameters for Sentinel-1
vis_params_s1_vv = ...

## . Helper Functions

## . Area of Interest (AOI)

In [12]:
# AOI (GeoJSON)
aoi_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [6.853409, 7.676968],
            [6.685181, 7.676968],
            [6.685181, 7.890588],
            [6.853409, 7.890588],
            [6.853409, 7.676968]
        ]
    ]
}

# Convert to GEE Feature
aoi_feature = ee.Feature(aoi_geojson)

# Default location to show the map
map_center_coords = 7.8175, 6.7730


In [32]:
# Show AOI on the map
aoi_map = geemap.Map(center=(7.8175, 6.7730), zoom=10)
aoi_map.add_basemap("SATELLITE")
aoi_map.addLayer(aoi_feature, vis_params_aoi, "AOI Flood")
aoi_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Get Admin Boundaries

In [ ]:
# Get Nigeria Admin Boundary from FAO

fao_bdry_l0 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0") # Adm0
fao_bdry_l1 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1") # Adm1
fao_bdry_l2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2") #Adm2

# Get the boundary of Nigeria from FAO GAUL

nga_bdry_l0 = fao_bdry_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # Main Nigerian bdry
nga_bdry_l1 = fao_bdry_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # States bdry
nga_bdry_l2 = fao_bdry_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # LGAs bdry

lokoja_by_name = nga_bdry_l2.filter(ee.Filter.eq("ADM2_NAME", "Lokoja"))

## . Download and Pre-Process Sentinel-2

In [41]:
# Function to download Sentinel-2
def download_s2(start_date: str, end_date : str, cloud_perc: int, extent : ee.Feature | ee.Geometry):
    """
    Download Sentinel-2 images filtered by date, cloud cover, and geographic extent.

    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        cloud_perc (int): Maximum allowable cloud pixel percentage.
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.

    Returns:
        ee.ImageCollection: The filtered Sentinel-2 collection.
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filterDate(start_date, end_date) # Dates filter
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_perc)) # Cloud filter
                    .filterBounds(extent.geometry())) # Boundary filter   

    img_count =  img_count = img_collection.size().getInfo() # Number of image in the collection

    print(f"There are {img_count} images in the Sentinel-2 image collection\n")
    
    return img_collection



# Function to apply cloud masking to Sentinel-2
# See reference: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY
def mask_clouds_s2(image: ee.Image, cloud_thresh: int):
    """
    Apply cloud masking using cloud probability
    Uses the S2_CLOUD_PROBABILITY dataset to mask out cloudy pixels.

    Args:
        image (ee.Image): Sentinel-2 image with cloud probability band
        cloud_thresh (int): Cloud probability threshold (0-100)

    Returns:
        ee.Image: The cloud-masked image.
    """
    clear_threshold = cloud_thresh
    clouds = ee.Image(
            image.get("cloud_mask")
            ).select("probability")
    
    no_clouds = clouds.lt(clear_threshold)
    masked_image = image.updateMask(no_clouds).copyProperties(image, ["system:time_start"])
    
    return masked_image
    

# Masks from the 20m and 60m bands for bad data at scene edges
def mask_edges_s2(image : ee.Image):
    """
    Mask bad data at scene edges using 20m and 60m band masks.
    
    Args:
        image: ee.Image - Sentinel-2 image
    
    Returns:
        ee.Image - Image with edges masked
    """
    masked_image = image.updateMask(
                    image.select("B8A").mask().updateMask(image.select("B9").mask())
                    ).copyProperties(image, ["system:time_start"])
    
    return masked_image 


# Function to convert Sentinel-2 DN to reflectance
def scale_bands_s2(image: ee.Image):
        """
        Convert Sentinel-2 digital numbers to surface reflectance.
    
    Args:
        image: ee.Image - Sentinel-2 image with DN values
    
    Returns:
        ee.Image - Image with scaled reflectance values (0-1)
        """
        scaled_image = image.multiply(0.0001).copyProperties(image, ["system:time_start"]) # Multiply image by 0.0001 and copy "system:time_start" property 
        return scaled_image


# Function to resample bands in Sentinel-2 to 10m
def resample_bands_s2(image: ee.Image):
        """
        Resample 20m bands to 10m resolution using bilinear interpolation.
    
    Args:
        image: ee.Image - Sentinel-2 image with 10m and 20m bands
    
    Returns:
        ee.Image - Image with all bands at 10m resolution
        """
        bands_10m = image.select(["B2", "B3", "B4", "B8"]) # Select 10m bands
        bands_20m = image.select(["B5", "B6", "B7", "B8A", "B11", "B12"]) # Select 20m bands 
        resampled_image = bands_20m.resample("bilinear").reproject(
                                                            crs = bands_10m.projection(), 
                                                             scale=10)
        
        combined_10m = bands_10m.addBands(resampled_image) # Combine original 10m with 20m
        
        return combined_10m.copyProperties(image, ["system:time_start"])


# Function to compute spectral indices
def compute_indices_s2(image:ee.Image):
        """
        Compute spectral indices (NDVI, NDWI, MNDWI, EVI) for Sentinel-2 imagery.
    
    Args:
        image: ee.Image - Sentinel-2 image with required bands
    
    Returns:
        ee.Image - Original image with added spectral indices
        """
        
        ndvi = image.normalizedDifference(["B8", "B4"]).rename("ndvi")
        ndwi = image.normalizedDifference(["B3", "B8"]).rename("ndwi")
        mndwi = image.normalizedDifference(["B3", "B11"]).rename("mndwi")
        evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR":image.select("B8"),
            "RED": image.select("B4"),
            "BLUE": image.select("B2")
        }
        ).rename("evi")
        
        updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(image, ["system:time_start"])
        
        return updated_image 



In [42]:
# Download Sentinel-2 images
s2_img_col = download_s2(after_start, after_end, cloud_perc, aoi_feature)

# Apply cloud masking to ALL IMAGES in the Sentinel-2 collection
# Cloud mask is typically applied using the cloud probability dataset
# edge masking and scaling
s2_img_col = s2_img_col.map(lambda img: mask_edges_s2(img))  # Edge masking
s2_img_col = s2_img_col.map(lambda img: scale_bands_s2(img)) # Scaling

# Resample bands for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: resample_bands_s2(img))

# Rescale pixel values  for ALL IMAGES in the Sentinel-2 collection


# Compute Indices for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: compute_indices_s2(img))

There are 15 images in the Sentinel-2 image collection



In [43]:
# Convert Sentinel-2 images to a List
s2_img_list = s2_img_col.toList(s2_img_col.size())

In [44]:
# Visualize Sentinel-2 imagery
# Create a median composite for visualization
s2_composite = s2_img_col.median().clip(aoi_feature)

aoi_map.addLayer(s2_composite, vis_params_s2_rgb, "Sentinel-2 RGB (Post-flood)")
aoi_map.addLayer(s2_composite, vis_params_s2_fcc, "Sentinel-2 False Color (Post-flood)")
aoi_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
aoi_map

Map(bottom=251197.0, center=[7.69942407690097, 6.735305786132813], controls=(WidgetControl(options=['position'…

## . Download and Pre-process Sentinel-1

In [ ]:
# Function to download Sentinel-1
def download_s1(start_date: str, end_date: str, extent: ee.Feature | ee.Geometry):
    """
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                        ... # Dates filter
                        ... # InstrumentMode
                        ... # Resolution filter
                        ... # Boundary filter
                         .select(["VV", "VH"]))

    # Number of images in the Sentinel-1 Collection
    img_count = ...

    print(f"There are {img_count} Sentinel-1 Images")


# Function to apply Speckle filter to Sentinel-1 (focal median)
def focal_speckle_filter(image : ee.Image, radius : int,  unit: str = "meters"):
    """
    Focal Median filter with circle kernel.
    """
    image_filtered = ....focal_median(radius, "circle", unit)
    image_filtered.copyProperties(image, image.propertyNames())

    return image_filtered

In [ ]:
# Download Sentinel-1 imagery
s1_img_col = ...

# Apply Speckle filter to ALL IMAGES in the Sentinel-1 collection
...


In [ ]:
# Convert Sentinel-1 images to a List
s1_img_list =

In [ ]:
# Visualize Sentinel-1 imagery

In [ ]:
s1_img_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterDate(after_start, after_end)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filterMetadata("resolution_meters", "equals", 10)
                .filterBounds(aoi_feature.geometry())
                .select(["VV", "VH"]))

# image.focal_median(radius, "circle", "meters")

## . Download Landsat-8

In [ ]:
def download_l8(start_date: str, end_date: str, extent: ee.Feature | ee.Geometry):
        """
        """
    
        # download and Filter and Landsat-8 image collection 
        img_collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                                  ... # Dates filter
                                  ... # Cloud filter
                                  ... # Boundary filter
                    )   

        img_count =  ... # Number of image in the collection

        print(f"There are {img_count} images in the Landsat-8 image collection\n")

        return img_collection 


# Band scaling factors for Landsat-8
def scale_factors_l8(image : ee.Image):
        """
        """

        optical_bands = image.select(...).multiply(...).add(...)  # scaling of reflectance values
        thermal_bands = image.select(...).multiply(...).add(...) # scaling of temp/thermal values

        scaled_bands =  image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)
        
        return scaled_bands


# Cloud Masking for Landsat-8
def mask_l8_sr(image : ee.Image):

        qa_mask = image.select("QA_PIXEL").bitwiseAnd(int("11111", 2)).eq(0)
        masked_image = image.updateMask(qa_mask).copyProperties(image, ["system:time_start"])


        return masked_image

# Function to compute spectral indices
def compute_indices_l8(image:ee.Image):
        """
        """
        .... # Use the example in Sentinel-2 above to complete this

In [ ]:
# Download Landsat-8 images
l8_img_col = download_l8(after_start, cloud_perc: int, extent : ee.Feature | ee.Geometry)

# Apply cloud masking to ALL IMAGES in the Landsat-8 collection
...


# Rescale pixel values  for ALL IMAGES in the Landsat-8 collection
...

# Compute Indices for ALL IMAGES in the SLandsat-8 collection
...

## . Download Building Dataset

In [ ]:
# Load Building Dataset (Google Open Buildings)
buildings = ee.FeatureCollection(
    "GOOGLE/Research/open-buildings-temporal/v1"
)

# Clip buildings to AOI
buildings_aoi = buildings.filterBounds(aoi_feature)

### . Boundary Visualization

In [37]:

# Add Nigeria boundaries
aoi_map.addLayer(nga_bdry_l0, {"color": "black"}, "NGA Bdry L0")
aoi_map.addLayer(nga_bdry_l1, {"color": "blue"}, "NGA State")
aoi_map.addLayer(nga_bdry_l2, {"color": "green"}, "NGA LGA L2")

# Add Lokoja
aoi_map.addLayer(lokoja_by_name, {"color": "black"}, "Lokoja (Name)")

# Display map
aoi_map

Map(bottom=251197.0, center=[7.69942407690097, 6.735305786132813], controls=(WidgetControl(options=['position'…

## . Flood Mapping

## . Damage / Impact Assessment